In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 10000

# Random assignment
group = np.random.choice(['A', 'B'], size=n, p=[0.5, 0.5])

# Conversion
conv_rate_A = 0.05
conv_rate_B = 0.065

converted = np.where(group == 'A',
                     np.random.binomial(1, conv_rate_A, n),
                     np.random.binomial(1, conv_rate_B, n))

# Time on site
time_A = np.random.normal(120, 30, n)
time_B = np.random.normal(130, 35, n)
time_on_site = np.where(group == 'A', time_A, time_B)
time_on_site = np.clip(time_on_site, 0, None)

# Pages viewed
pages_A = np.random.poisson(4, n)
pages_B = np.random.poisson(4.3, n)
pages_viewed = np.where(group == 'A', pages_A, pages_B)

# DataFrame
df = pd.DataFrame({
    'visitor_id': range(1, n+1),
    'group': group,
    'converted': converted,
    'time_on_site': np.round(time_on_site, 1),
    'pages_viewed': pages_viewed
})

df.head()

,visitor_id,group,converted,time_on_site,pages_viewed
0,1,A,0,101.1,3
1,2,B,0,157.6,4
2,3,B,0,128.5,2
3,4,B,0,87.6,3
4,5,A,0,109.7,3


# Hypotheses

## Conversion Rate
H₀: p_B ≤ p_A  
H₁: p_B > p_A  

## Time on Site
H₀: μ_B ≤ μ_A  
H₁: μ_B > μ_A  

## Pages Viewed
H₀: μ_B ≤ μ_A  
H₁: μ_B > μ_A  

Significance level α = 0.05

In [7]:
from statsmodels.stats.proportion import proportions_ztest

count = [
    df[df.group=='B']['converted'].sum(),
    df[df.group=='A']['converted'].sum()
]

nobs = [
    df[df.group=='B'].shape[0],
    df[df.group=='A'].shape[0]
]

z_stat, p_value = proportions_ztest(count, nobs, alternative='larger')

print("Z-statistic:", z_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject H₀: Treatment B improves conversion.")
else:
    print("Fail to reject H₀.")

Z-statistic: 2.2551513377591643
P-value: 0.012061915588574583
Reject H₀: Treatment B improves conversion.


In [9]:
from scipy import stats

time_A = df[df.group=='A']['time_on_site']
time_B = df[df.group=='B']['time_on_site']

t_stat, p_value = stats.ttest_ind(time_B, time_A, equal_var=False)

p_one_tailed = p_value / 2

print("t-statistic:", t_stat)
print("One-tailed p-value:", p_one_tailed)

if p_one_tailed < 0.05:
    print("Reject H₀: Treatment B increases time on site.")
else:
    print("Fail to reject H₀.")

t-statistic: 15.950200206182469
One-tailed p-value: 7.438228279545056e-57
Reject H₀: Treatment B increases time on site.


In [11]:
pages_A = df[df.group=='A']['pages_viewed']
pages_B = df[df.group=='B']['pages_viewed']

t_stat, p_value = stats.ttest_ind(pages_B, pages_A, equal_var=False)

p_one_tailed = p_value / 2

print("t-statistic:", t_stat)
print("One-tailed p-value:", p_one_tailed)

if p_one_tailed < 0.05:
    print("Reject H₀: Treatment B increases pages viewed.")
else:
    print("Fail to reject H₀.")

t-statistic: 6.758681729677644
One-tailed p-value: 7.353060180421787e-12
Reject H₀: Treatment B increases pages viewed.


In [13]:
import numpy as np

# Conversion Lift
pA = count[1]/nobs[1]
pB = count[0]/nobs[0]
print("Conversion Lift:", pB - pA)

# Cohen's d (time)
d_time = (np.mean(time_B) - np.mean(time_A)) / np.std(time_A, ddof=1)
print("Cohen's d (Time on Site):", d_time)

# Cohen's d (pages)
d_pages = (np.mean(pages_B) - np.mean(pages_A)) / np.std(pages_A, ddof=1)
print("Cohen's d (Pages Viewed):", d_pages)

Conversion Lift: 0.010577803895812095
Cohen's d (Time on Site): 0.34677101294693036
Cohen's d (Pages Viewed): 0.13814198894265484


# Final Conclusion

At α = 0.05:

- Conversion rate is significantly higher for Treatment B.
- Time on site is significantly higher for Treatment B.
- Pages viewed is significantly higher for Treatment B.

All null hypotheses are rejected.

Business Recommendation:
The new website design (Treatment B) should be fully implemented.